In [ ]:
import pandas as pd
import numpy as np
import os

# 1. Load the big file
INPUT_FILE = 'data/raw/filtered_2024_2026.csv'
print(f"📂 Loading {INPUT_FILE}...")
df = pd.read_csv(INPUT_FILE)
df.columns = [c.lower() for c in df.columns]

# 2. Define the number of parts (5 members)
num_parts = 5
total_rows = len(df)
chunk_size = int(np.ceil(total_rows / num_parts))

# 3. Create the output folder
if not os.path.exists('data/chunks'): 
    os.makedirs('data/chunks')

# 4. Split and Save using .iloc (Guaranteed to remain a DataFrame)
print(f"✂️ Splitting {total_rows} rows into {num_parts} parts...")

for i in range(num_parts):
    start_idx = i * chunk_size
    end_idx = min((i + 1) * chunk_size, total_rows)
    
    # Extract the chunk
    part = df.iloc[start_idx:end_idx]
    
    # Save the chunk
    filename = f'data/chunks/work_part_{i+1}.csv'
    part.to_csv(filename, index=False)
    
    print(f"✅ Created {filename} | Rows: {len(part)}")

print("\n🎉 Split Complete! Share these 5 files with your team.")

In [ ]:
# Phase 1B: Parallel Worker Script
import pandas as pd
import numpy as np
import requests
import time
import os
import reverse_geocoder as rg
from global_land_mask import globe

# --- ASSIGNMENT: Change these two lines for your assigned part ---
MY_PART = 1  # Change to 1, 2, 3, 4, or 5
INPUT_FILE = f'data/chunks/work_part_{MY_PART}.csv'
OUTPUT_FILE = f'data/chunks/finished_part_{MY_PART}.csv'
# ----------------------------------------------------------------

HISTORICAL_API = "https://archive-api.open-meteo.com/v1/archive"

# 1. LOAD & PRE-FILTER (Strict India)
print(f"🚀 Loading Part {MY_PART}...")
df = pd.read_csv(INPUT_FILE)
df = df[(df['latitude'] >= 6) & (df['latitude'] <= 38) & (df['longitude'] >= 68) & (df['longitude'] <= 98)].copy()

# Filter for India Borders
print("🧐 Filtering for India borders...")
coords = list(zip(df['latitude'], df['longitude']))
df['country'] = [x['cc'] for x in rg.search(coords)]
df = df[df['country'] == 'IN'].copy()
df['acq_date'] = pd.to_datetime(df['acq_date']).dt.strftime('%Y-%m-%d')
print(f"✅ Verified {len(df)} points inside India.")

# 2. CHECKPOINT LOGIC (Resume if stopped)
if os.path.exists(OUTPUT_FILE):
    df_done = pd.read_csv(OUTPUT_FILE)
    done_keys = set(df_done['latitude'].astype(str) + df_done['acq_date'].astype(str))
    print(f"🔄 Resuming... {len(done_keys)} rows already finished.")
else:
    done_keys = set()
    df.head(0).to_csv(OUTPUT_FILE, index=False)

# 3. TURBO BATCH RETRIEVAL
print("☁️ Fetching weather data (50 points per call)...")
grouped = df.groupby('acq_date')

for date_str, group in grouped:
    # Skip rows already done
    to_proc = group[~(group['latitude'].astype(str) + date_str).isin(done_keys)]
    if to_proc.empty: continue

    # Batch process 50 locations per API call
    for j in range(0, len(to_proc), 50):
        batch = to_proc.iloc[j : j + 50]
        params = {
            "latitude": ",".join(map(str, batch['latitude'])),
            "longitude": ",".join(map(str, batch['longitude'])),
            "start_date": date_str, "end_date": date_str,
            "hourly": "temperature_2m,relative_humidity_2m,wind_speed_10m"
        }
        
        try:
            r = requests.get(HISTORICAL_API, params=params, timeout=60)
            if r.status_code == 200:
                data = r.json()
                results = data if isinstance(data, list) else [data]
                enriched = []
                for idx, (_, row) in enumerate(batch.iterrows()):
                    w = results[idx]['hourly']
                    enriched.append({
                        **row.to_dict(),
                        'temperature_2m': w['temperature_2m'][12], # Noon data
                        'relative_humidity_2m': w['relative_humidity_2m'][12],
                        'wind_speed_10m': w['wind_speed_10m'][12]
                    })
                pd.DataFrame(enriched).to_csv(OUTPUT_FILE, mode='a', header=False, index=False)
            elif r.status_code == 429:
                print("🛑 Rate Limit! Waiting 60s...")
                time.sleep(30)
        except Exception as e:
            print(f"⚠️ Error: {e}")
        
        time.sleep(1) # Polite delay
    print(f"✅ Date {date_str} finished.", end='\r')

print(f"\n🎉 Part {MY_PART} is COMPLETE!")